# ClusterEnv — GRPO Scheduler Training

**Environment:** Power-Capped AI Cluster Scheduling Under Information Asymmetry  
**Agent:** Qwen2.5-3B-Instruct-bnb-4bit fine-tuned with GRPO (Group Relative Policy Optimization)  
**Task:** An LLM scheduler learns to allocate compute jobs across competing teams while one team systematically inflates priority claims.

## Prerequisites
- Runtime: **GPU** (T4 or better). Go to Runtime → Change runtime type → GPU.
- No credentials needed. All models and code are public.

## What this notebook does
1. Installs dependencies (Unsloth + ClusterEnv)
2. Downloads the pre-trained PPO cooling controller from HF Hub
3. Runs GRPO training for `N_ITERATIONS` iterations
4. Saves reward/loss curves and the trained LoRA adapter

> For judges re-running: the default config runs 10 iterations (~20 min on T4) to show convergence direction. Increase `N_ITERATIONS` to 30 for a full run.

## Step 1 — Install dependencies

In [ ]:
%%capture
# Unsloth first (owns the torch version)
!pip install unsloth
# Everything else
!pip install stable-baselines3 gymnasium huggingface_hub trl accelerate bitsandbytes xformers matplotlib

## Step 2 — Clone the ClusterEnv repository

In [ ]:
import os

REPO_URL    = "https://github.com/DrishyaShah/datacenter-env.git"
REPO_BRANCH = "arhaan/finale-v1"   # branch with all ClusterEnv code
REPO_DIR    = "/content/datacenter-env"

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR}")

os.chdir(REPO_DIR)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())

## Step 3 — Download PPO cooling controller from HF Hub

In [ ]:
from huggingface_hub import hf_hub_download

ppo_dest = "training/cooling_controller_best/best_model.zip"
os.makedirs(os.path.dirname(ppo_dest), exist_ok=True)

if not os.path.exists(ppo_dest):
    hf_hub_download(
        repo_id   = "Mephisto2412/clusterenv-ppo-cooling",
        filename  = "best_model.zip",
        local_dir = "training/cooling_controller_best",
    )
    print("PPO model downloaded.")
else:
    print("PPO model already present.")

## Step 4 — Verify the environment works

In [ ]:
from server.cluster_environment import ClusterEnvironment
from server.agents.baseline_scheduler import priority_weighted_threshold

env = ClusterEnvironment(enable_chiller_fault=False)
obs = env.reset(seed=0)
decisions = priority_weighted_threshold(obs)
obs2, reward, done, info = env.step(decisions)

print(f"Environment OK")
print(f"  Window 0 reward   : {reward:+.4f}")
print(f"  Window 1 state    : window_idx={obs2.window_idx}")
print(f"  Oversight flags   : {len(obs2.oversight_flags)} (Team B gaming detected)")

## Step 5 — Configure training

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Judges re-running: N_ITERATIONS=10 takes ~20 min on T4, shows convergence direction.
# Full run used N_ITERATIONS=30 on A10G (~2 hours).

MODEL_NAME       = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH   = 4096
N_ITERATIONS     = 10      # increase to 30 for full run
G_EPISODES       = 2       # episodes per iteration (2 = fast verification, 4 = full)
LEARNING_RATE    = 1e-5
TEMPERATURE      = 0.7
MAX_NEW_TOKENS   = 768
LORA_R           = 16
LORA_ALPHA       = 32
ADAPTER_DIR      = "training/grpo_adapter_colab"
CHECKPOINT_EVERY = 5

print(f"Model          : {MODEL_NAME}")
print(f"Iterations     : {N_ITERATIONS}")
print(f"Episodes/iter  : {G_EPISODES}  -> {G_EPISODES * 8} samples/iter")
print(f"Adapter output : {ADAPTER_DIR}")

## Step 6 — Load model with Unsloth

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout   = 0.0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 7 — Run GRPO training

In [ ]:
import numpy as np
from collections import defaultdict
from torch.optim import AdamW
from training.rollout import collect_rollouts, compute_grpo_advantages
from training.train_grpo import compute_log_prob, make_generate_fn

optimizer   = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
os.makedirs(ADAPTER_DIR, exist_ok=True)

reward_log, loss_log = [], []
generate_fn = make_generate_fn(model, tokenizer, TEMPERATURE, MAX_NEW_TOKENS)

for iteration in range(N_ITERATIONS):
    # — Rollout phase —
    rollouts   = collect_rollouts(generate_fn, n_episodes=G_EPISODES,
                                  base_seed=iteration * G_EPISODES,
                                  enable_chiller_fault=False)
    advantages = compute_grpo_advantages(rollouts)

    # — Gradient phase —
    FastLanguageModel.for_training(model)
    optimizer.zero_grad()
    total_loss = 0.0
    for sample, adv in zip(rollouts, advantages):
        lp   = compute_log_prob(model, tokenizer, sample["prompt"], sample["completion"])
        loss = (-adv * lp) / len(rollouts)
        loss.backward()
        total_loss += loss.item()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()

    # — Logging —
    mean_reward    = float(np.mean([r["reward"] for r in rollouts]))
    parse_fails    = sum(1 for r in rollouts if r["reward"] <= -0.4)
    win_groups: dict[int, list[float]] = defaultdict(list)
    for r in rollouts:
        win_groups[r["window_idx"]].append(r["reward"])
    group_std = float(np.mean([np.std(v) for v in win_groups.values()]))

    reward_log.append(mean_reward)
    loss_log.append(total_loss)

    print(f"[{iteration+1:3d}/{N_ITERATIONS}]  "
          f"loss={total_loss:+.4f}  reward={mean_reward:+.4f}  "
          f"group_std={group_std:.3f}  grad={grad_norm:.3f}  "
          f"parse_fail={parse_fails}/{len(rollouts)}")

    if iteration < 2:
        preview = rollouts[0]["completion"].replace("\n", " ")[:100]
        print(f"  sample: {preview!r}")

    if (iteration + 1) % CHECKPOINT_EVERY == 0:
        ckpt = os.path.join(ADAPTER_DIR, f"ckpt_{iteration+1}")
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f"  Checkpoint -> {ckpt}")

print("\nTraining complete.")

## Step 8 — Save training curves

In [ ]:
import matplotlib.pyplot as plt

iters = list(range(1, len(reward_log) + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(iters, reward_log, color="#2196F3", linewidth=2, marker="o", markersize=5)
ax1.axhline(0.28, color="gray", linestyle="--", linewidth=1, label="rule-based baseline (+0.28)")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Mean episode reward")
ax1.set_title("GRPO Reward Curve — ClusterEnv Scheduler")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(iters, loss_log, color="#F44336", linewidth=2, marker="o", markersize=5)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("GRPO loss")
ax2.set_title("GRPO Loss Curve — ClusterEnv Scheduler")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training/grpo_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training/grpo_training_curves.png")

## Step 9 — Save final adapter

In [ ]:
final_path = os.path.join(ADAPTER_DIR, "final")
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"Adapter saved -> {final_path}")

# Optional: push to HF Hub
# from huggingface_hub import login
# login()  # enter your token
# model.push_to_hub("your-username/clusterenv-grpo-adapter")
# tokenizer.push_to_hub("your-username/clusterenv-grpo-adapter")